# ДЗ 05. Feature matching / Homework 05

**RU.** Реальные пары фотографий Oxford Affine Covariant Features. Наблюдение — пара кадров одной сцены; признаки — координаты ключевых точек и SIFT/ORB-дескрипторы; целевая разметка — гомография H1→j. `graf` и `leuven` служат для демонстрации и настройки. `wall` и `bikes` остаются для итоговой проверки после фиксации параметров. Кадры одной сцены нельзя случайно распределять между настройкой и тестом. Это учебное разбиение четырёх сцен, не официальный benchmark и не оценка всей популяции сцен.

**EN.** Real photograph pairs from Oxford Affine Covariant Features. An observation is a pair of views of one scene. Features are keypoint coordinates and SIFT/ORB descriptors. The reference target is the H1→j homography. Use `graf` and `leuven` for demonstrations and tuning. Reserve `wall` and `bikes` for evaluation after freezing settings. Never randomly split frames of the same scene across tuning and evaluation. This four-scene teaching split is not the official benchmark or an estimate over all scenes.

**Источники / Sources:** [Oxford dataset](https://www.robots.ox.ac.uk/~vgg/research/affine/), [Data description](https://www.robots.ox.ac.uk/~vgg/research/affine/det_eval_files/DataREADME), Mikolajczyk et al., IJCV 2005; [OpenCV matching](https://docs.opencv.org/4.x/dc/dc3/tutorial_py_matcher.html).

**Условия / Terms.** Данные доступны на странице авторов для оценки методов; отдельная открытая лицензия на фотографии там не указана. Сохраняйте ссылки и авторство, не приписывайте фотографиям MIT-лицензию кода. Ноутбук загружает оригиналы с Oxford, изображения в репозиторий не включены. / The authors provide data for method evaluation; the page does not state a separate open image license. Retain attribution and sources. Do not apply a code MIT license to photographs. The notebook downloads original data from Oxford; images are not bundled in the repository.

**Ограничения / Limits.** Плоские сцены или фиксированный центр камеры. Гомография не моделирует произвольный параллакс. Эталонные H имеют погрешность. / Planar scenes or a fixed camera centre. A homography does not model arbitrary parallax. Reference homographies have uncertainty.

**Среда / Environment.** Python 3.11+, CPU, NumPy 2.2.6, OpenCV 4.12.0.88. Четыре архива около 33 MB. SHA-256 проверяется, синтетического fallback нет. / Four archives total about 33 MB. Downloads are checked with SHA-256. There is no synthetic fallback.


**ID:** `Feature_Matching_Autograder`, версия / version 2, максимум / maximum 100.
RU: реализуйте четыре функции. Не меняйте проверочные ячейки. Прогоните Restart & Run All и сдайте .ipynb с именем и группой. Исходный балл не включает штраф. Сроки по группам: [матрица](https://docs.google.com/spreadsheets/d/15N5GGHujXPAzRNuyOfmooT7A7sKhXgrMfWHkUkvYYiM/edit). Пустой срок требует назначения преподавателем. Штраф за просрочку рассчитывается внешним проверяющим по политике курса; срок уже включает запас. После этого при отсутствии отметки практики лекции 5 итог ×0.9. Время берёт проверяющий из Drive, не из ноутбука.
EN: implement four functions. Preserve grader cells. Restart & Run All and submit the notebook with name and group. Output raw points only. Use the linked group deadline matrix. A blank deadline needs an instructor date. The external checker applies the course lateness policy; the deadline already includes grace. Then multiply the remainder by 0.9 if lecture 5 practice is unmarked. The checker uses trusted Drive timestamps, never notebook time.

Старый файл остаётся в Archive; ошибочный ID `HLP_AvoidableBias` не используется. / The legacy file stays in Archive; its incorrect ID `HLP_AvoidableBias` is not used.

In [ ]:
full_name = ""
student_group = ""
assignment_id = "Feature_Matching_Autograder"

In [ ]:
# Colab: run once if necessary, then restart the runtime if requested.
# Colab: при необходимости выполните и перезапустите среду.
# %pip install -q numpy==2.2.6 opencv-python-headless==4.12.0.88 matplotlib


In [ ]:
DATA_MANIFEST = {'graf': {'url': 'https://www.robots.ox.ac.uk/~vgg/research/affine/det_eval_files/graf.tar.gz', 'sha256': '999871b945ee968a00a0d5f9af957d1382fb9dae1511cdee9553366817b53b5b', 'bytes': 8414417}, 'leuven': {'url': 'https://www.robots.ox.ac.uk/~vgg/research/affine/det_eval_files/leuven.tar.gz', 'sha256': '9b1ade6e64e5b27eee43463ff8f9ad2bd2f9c6911666e93de2634f3a5c0fe5dc', 'bytes': 6739237}, 'wall': {'url': 'https://www.robots.ox.ac.uk/~vgg/research/affine/det_eval_files/wall.tar.gz', 'sha256': '3cb3df75f40471f093adf6647d13df6a9f4312f7a222fc591d03f6c08e086344', 'bytes': 9991441}, 'bikes': {'url': 'https://www.robots.ox.ac.uk/~vgg/research/affine/det_eval_files/bikes.tar.gz', 'sha256': '3b95c49accadfd71e63c2fa0f7602abcbadd87d7f53ff54eb1efb15210d19e0c', 'bytes': 7366360}}
from pathlib import Path
import hashlib, io, json, tarfile, urllib.request
import numpy as np
import cv2
cv2.setNumThreads(1)
cv2.setRNGSeed(5102026)
DATA = Path('data')
MANIFEST = DATA_MANIFEST

def load_sequence(name):
    spec=MANIFEST[name]; archive=DATA/(name+'.tar.gz'); folder=DATA/name
    DATA.mkdir(exist_ok=True)
    if not archive.exists():
        with urllib.request.urlopen(spec['url'],timeout=120) as response: blob=response.read()
        if hashlib.sha256(blob).hexdigest()!=spec['sha256']: raise RuntimeError('Dataset checksum mismatch')
        archive.write_bytes(blob)
    if hashlib.sha256(archive.read_bytes()).hexdigest()!=spec['sha256']: raise RuntimeError('Dataset checksum mismatch')
    folder.mkdir(exist_ok=True)
    with tarfile.open(archive) as tar:
        for member in tar.getmembers():
            if not member.isfile(): continue
            # Only known flat dataset assets, never arbitrary archive paths.
            leaf=Path(member.name).name
            if leaf not in {*(f'img{i}.ppm' for i in range(1,7)),*(f'H1to{i}p' for i in range(2,7))}:continue
            target=folder/leaf
            if not target.exists():target.write_bytes(tar.extractfile(member).read())
    return folder

def pair(sequence, index=2):
    folder=load_sequence(sequence)
    a=cv2.imread(str(folder/'img1.ppm'),cv2.IMREAD_GRAYSCALE)
    b=cv2.imread(str(folder/f'img{index}.ppm'),cv2.IMREAD_GRAYSCALE)
    if a is None or b is None:raise RuntimeError('Real dataset images unavailable')
    H=np.loadtxt(folder/f'H1to{index}p')
    # Official Oxford repeatability.m specifies coordinates 0..width.
    # Use H1toj directly with OpenCV coordinates; do not add a 1-pixel shift.
    return a,b,H/H[2,2]

def features(image,method='SIFT'):
    detector=cv2.SIFT_create(nfeatures=1500) if method=='SIFT' else cv2.ORB_create(nfeatures=1500,WTA_K=2)
    return detector.detectAndCompute(image,None)

def neighbors(d1,d2,method='SIFT'):
    if d1 is None or d2 is None or len(d1)==0 or len(d2)<2:return []
    norm=cv2.NORM_L2 if method=='SIFT' else cv2.NORM_HAMMING
    return cv2.BFMatcher(norm).knnMatch(d1,d2,k=2)







## ratio_filter — 20/100
Для каждой пары DMatch верните ближайший match, только если d1 < ratio*d2. Пары с числом соседей не 2 пропускайте. ratio вне (0,1) вызывает ValueError. Вход не менять. / Return the nearest match only when d1 < ratio*d2. Skip rows without exactly two neighbours. Raise ValueError outside (0,1). Do not mutate input.

In [ ]:
def ratio_filter(knn, ratio=.75):
    raise NotImplementedError("Ваш код / Your code")

## estimate_homography — 30/100
Верните (H, bool mask длины M), H отображает изображение 1 в 2. Используйте RANSAC, maxIters=5000, confidence=.995, seed=5102026. При M<4 или невозможности оценки верните (None, все False). Неположительный threshold вызывает ValueError. / Return (H, Boolean mask of length M), mapping image 1 to 2. Use RANSAC with the stated settings. For fewer than four matches or estimation failure return (None, all False). Reject nonpositive thresholds.

In [ ]:
def estimate_homography(kp1, kp2, matches, threshold=3.):
    raise NotImplementedError("Ваш код / Your code")

## transfer_errors — 25/100
Для p,q формы (N,2) верните вектор ||dehom(Hp)-q||. Несовпадающие формы вызывают ValueError, нулевой знаменатель (abs<=1e-12) или нечисловая точка даёт inf. Для N=0 верните пустой вектор. / Return forward transfer errors for paired (N,2) arrays. Reject mismatched shapes. Invalid coordinates or near-zero homogeneous denominators yield infinity. Empty input yields an empty vector.

In [ ]:
def transfer_errors(p, q, H):
    raise NotImplementedError("Ваш код / Your code")

## choose_config — 25/100
Используйте только split=validation и конечные precision. Усредните precision по парам для каждого config, верните ID с максимальным средним, при равенстве — минимальный лексикографически. Без валидных записей ValueError. / Use only validation rows with finite precision. Macro-average over pairs per config. Return the highest mean, breaking ties lexicographically. Raise ValueError if no valid rows remain.

In [ ]:
def choose_config(records):
    raise NotImplementedError("Ваш код / Your code")

## Проверка на реальных данных / Tests on real data
RU: служебные ячейки не редактировать. Тестовые параметры здесь фиксированы преподавателем, это не подбор по test. / Do not edit these cells. The instructor fixes these test settings; they are not tuning on the held-out scenes.

In [ ]:
a,b,GT=pair('graf',2)
KP1,D1=features(a,'SIFT');KP2,D2=features(b,'SIFT')
KNN=neighbors(D1,D2,'SIFT')
EXPECTED=[m for row in KNN if len(row)==2 for m,n in [sorted(row,key=lambda x:x.distance)] if m.distance < .75*n.distance]
P=np.float64([KP1[m.queryIdx].pt for m in EXPECTED]);Q=np.float64([KP2[m.trainIdx].pt for m in EXPECTED])


In [ ]:
VALIDATION_RECORDS = [{'sequence': 'graf', 'pair': '1-2', 'method': 'SIFT', 'ratio': 0.6, 'matches': 578, 'inliers': 570, 'precision': 0.9757785467128027, 'gt_median_px': 0.448135006214046, 'homography_found': True, 'split': 'validation', 'config': 'SIFT:0.6'}, {'sequence': 'graf', 'pair': '1-4', 'method': 'SIFT', 'ratio': 0.6, 'matches': 22, 'inliers': 13, 'precision': 0.5909090909090909, 'gt_median_px': 1.8663648357375, 'homography_found': True, 'split': 'validation', 'config': 'SIFT:0.6'}, {'sequence': 'graf', 'pair': '1-2', 'method': 'SIFT', 'ratio': 0.75, 'matches': 673, 'inliers': 623, 'precision': 0.9138187221396731, 'gt_median_px': 0.4858040722757424, 'homography_found': True, 'split': 'validation', 'config': 'SIFT:0.75'}, {'sequence': 'graf', 'pair': '1-4', 'method': 'SIFT', 'ratio': 0.75, 'matches': 83, 'inliers': 34, 'precision': 0.40963855421686746, 'gt_median_px': 8.690079093598717, 'homography_found': True, 'split': 'validation', 'config': 'SIFT:0.75'}, {'sequence': 'graf', 'pair': '1-2', 'method': 'SIFT', 'ratio': 0.9, 'matches': 868, 'inliers': 658, 'precision': 0.7476958525345622, 'gt_median_px': 0.7144998830596117, 'homography_found': True, 'split': 'validation', 'config': 'SIFT:0.9'}, {'sequence': 'graf', 'pair': '1-4', 'method': 'SIFT', 'ratio': 0.9, 'matches': 408, 'inliers': 80, 'precision': 0.2034313725490196, 'gt_median_px': 153.41587293248352, 'homography_found': True, 'split': 'validation', 'config': 'SIFT:0.9'}, {'sequence': 'graf', 'pair': '1-2', 'method': 'ORB', 'ratio': 0.6, 'matches': 311, 'inliers': 303, 'precision': 0.9581993569131833, 'gt_median_px': 0.9421832159418523, 'homography_found': True, 'split': 'validation', 'config': 'ORB:0.6'}, {'sequence': 'graf', 'pair': '1-4', 'method': 'ORB', 'ratio': 0.6, 'matches': 1, 'inliers': 0, 'precision': 0.0, 'gt_median_px': 12.497417826497747, 'homography_found': False, 'split': 'validation', 'config': 'ORB:0.6'}, {'sequence': 'graf', 'pair': '1-2', 'method': 'ORB', 'ratio': 0.75, 'matches': 565, 'inliers': 534, 'precision': 0.9362831858407079, 'gt_median_px': 1.0447032847164757, 'homography_found': True, 'split': 'validation', 'config': 'ORB:0.75'}, {'sequence': 'graf', 'pair': '1-4', 'method': 'ORB', 'ratio': 0.75, 'matches': 36, 'inliers': 12, 'precision': 0.3611111111111111, 'gt_median_px': 13.251542862827907, 'homography_found': True, 'split': 'validation', 'config': 'ORB:0.75'}, {'sequence': 'graf', 'pair': '1-2', 'method': 'ORB', 'ratio': 0.9, 'matches': 956, 'inliers': 795, 'precision': 0.8420502092050209, 'gt_median_px': 1.1887662926424698, 'homography_found': True, 'split': 'validation', 'config': 'ORB:0.9'}, {'sequence': 'graf', 'pair': '1-4', 'method': 'ORB', 'ratio': 0.9, 'matches': 353, 'inliers': 86, 'precision': 0.23796033994334279, 'gt_median_px': 112.22862832180999, 'homography_found': True, 'split': 'validation', 'config': 'ORB:0.9'}, {'sequence': 'leuven', 'pair': '1-2', 'method': 'SIFT', 'ratio': 0.6, 'matches': 724, 'inliers': 708, 'precision': 0.9779005524861878, 'gt_median_px': 0.25499416267795016, 'homography_found': True, 'split': 'validation', 'config': 'SIFT:0.6'}, {'sequence': 'leuven', 'pair': '1-4', 'method': 'SIFT', 'ratio': 0.6, 'matches': 530, 'inliers': 515, 'precision': 0.9735849056603774, 'gt_median_px': 0.40490716766833457, 'homography_found': True, 'split': 'validation', 'config': 'SIFT:0.6'}, {'sequence': 'leuven', 'pair': '1-2', 'method': 'SIFT', 'ratio': 0.75, 'matches': 801, 'inliers': 756, 'precision': 0.9438202247191011, 'gt_median_px': 0.26597823737878656, 'homography_found': True, 'split': 'validation', 'config': 'SIFT:0.75'}, {'sequence': 'leuven', 'pair': '1-4', 'method': 'SIFT', 'ratio': 0.75, 'matches': 638, 'inliers': 584, 'precision': 0.9184952978056427, 'gt_median_px': 0.4374558085147113, 'homography_found': True, 'split': 'validation', 'config': 'SIFT:0.75'}, {'sequence': 'leuven', 'pair': '1-2', 'method': 'SIFT', 'ratio': 0.9, 'matches': 968, 'inliers': 799, 'precision': 0.8264462809917356, 'gt_median_px': 0.3062938308398824, 'homography_found': True, 'split': 'validation', 'config': 'SIFT:0.9'}, {'sequence': 'leuven', 'pair': '1-4', 'method': 'SIFT', 'ratio': 0.9, 'matches': 838, 'inliers': 637, 'precision': 0.7577565632458234, 'gt_median_px': 0.5333700431335282, 'homography_found': True, 'split': 'validation', 'config': 'SIFT:0.9'}, {'sequence': 'leuven', 'pair': '1-2', 'method': 'ORB', 'ratio': 0.6, 'matches': 497, 'inliers': 483, 'precision': 0.971830985915493, 'gt_median_px': 0.7402241958447328, 'homography_found': True, 'split': 'validation', 'config': 'ORB:0.6'}, {'sequence': 'leuven', 'pair': '1-4', 'method': 'ORB', 'ratio': 0.6, 'matches': 273, 'inliers': 263, 'precision': 0.9523809523809523, 'gt_median_px': 0.8750716200800677, 'homography_found': True, 'split': 'validation', 'config': 'ORB:0.6'}, {'sequence': 'leuven', 'pair': '1-2', 'method': 'ORB', 'ratio': 0.75, 'matches': 704, 'inliers': 661, 'precision': 0.9460227272727273, 'gt_median_px': 0.8074634005763146, 'homography_found': True, 'split': 'validation', 'config': 'ORB:0.75'}, {'sequence': 'leuven', 'pair': '1-4', 'method': 'ORB', 'ratio': 0.75, 'matches': 466, 'inliers': 423, 'precision': 0.8969957081545065, 'gt_median_px': 1.0592977165807889, 'homography_found': True, 'split': 'validation', 'config': 'ORB:0.75'}, {'sequence': 'leuven', 'pair': '1-2', 'method': 'ORB', 'ratio': 0.9, 'matches': 992, 'inliers': 832, 'precision': 0.8407258064516129, 'gt_median_px': 1.034214652154557, 'homography_found': True, 'split': 'validation', 'config': 'ORB:0.9'}, {'sequence': 'leuven', 'pair': '1-4', 'method': 'ORB', 'ratio': 0.9, 'matches': 792, 'inliers': 609, 'precision': 0.7702020202020202, 'gt_median_px': 1.364365030519916, 'homography_found': True, 'split': 'validation', 'config': 'ORB:0.9'}, {'sequence': 'wall', 'pair': '1-2', 'method': 'SIFT', 'ratio': 0.6, 'matches': 677, 'inliers': 677, 'precision': 1.0, 'gt_median_px': 0.992959416368124, 'homography_found': True, 'split': 'test', 'config': 'SIFT:0.6'}, {'sequence': 'wall', 'pair': '1-4', 'method': 'SIFT', 'ratio': 0.6, 'matches': 126, 'inliers': 124, 'precision': 0.9365079365079365, 'gt_median_px': 1.0845924320249627, 'homography_found': True, 'split': 'test', 'config': 'SIFT:0.6'}, {'sequence': 'wall', 'pair': '1-6', 'method': 'SIFT', 'ratio': 0.6, 'matches': 0, 'inliers': 0, 'precision': 0.0, 'gt_median_px': None, 'homography_found': False, 'split': 'test', 'config': 'SIFT:0.6'}, {'sequence': 'bikes', 'pair': '1-2', 'method': 'SIFT', 'ratio': 0.6, 'matches': 387, 'inliers': 373, 'precision': 0.958656330749354, 'gt_median_px': 0.3603167520878558, 'homography_found': True, 'split': 'test', 'config': 'SIFT:0.6'}, {'sequence': 'bikes', 'pair': '1-4', 'method': 'SIFT', 'ratio': 0.6, 'matches': 180, 'inliers': 169, 'precision': 0.9333333333333333, 'gt_median_px': 0.5961177996826297, 'homography_found': True, 'split': 'test', 'config': 'SIFT:0.6'}, {'sequence': 'bikes', 'pair': '1-6', 'method': 'SIFT', 'ratio': 0.6, 'matches': 113, 'inliers': 95, 'precision': 0.7964601769911505, 'gt_median_px': 1.6671182015024117, 'homography_found': True, 'split': 'test', 'config': 'SIFT:0.6'}]

In [ ]:
REFERENCE_SELECTION = 'SIFT:0.6'
def _ids(matches):return [(m.queryIdx,m.trainIdx,round(m.distance,5)) for m in matches]
def _must_raise(fn):
    try:fn()
    except ValueError:return
    raise AssertionError('Expected ValueError')
def test_ratio():
    for tau in [.6,.75,.9]:
        wanted=[m for row in KNN if len(row)==2 for m,n in [sorted(row,key=lambda x:x.distance)] if m.distance<tau*n.distance]
        assert _ids(ratio_filter(KNN,tau))==_ids(wanted)
    assert ratio_filter([], .75)==[]
    assert ratio_filter([KNN[0][:1]],.75)==[]
    _must_raise(lambda:ratio_filter(KNN,1.))
def test_homography():
    for seq in ['graf','wall']:
        a,b,gt=pair(seq,2); k1,d1=features(a); k2,d2=features(b)
        good=[m for row in neighbors(d1,d2) if len(row)==2 for m,n in [row] if m.distance<.75*n.distance]
        H,mask=estimate_homography(k1,k2,good)
        assert H is not None and H.shape==(3,3) and np.isfinite(H).all()
        assert isinstance(mask,np.ndarray) and mask.dtype==bool and mask.shape==(len(good),)
        assert mask.sum()>=10
        p=np.float64([k1[m.queryIdx].pt for m in good]);q=np.float64([k2[m.trainIdx].pt for m in good])
        projected=cv2.perspectiveTransform(p.reshape(-1,1,2),H).reshape(-1,2)
        assert np.mean(np.linalg.norm(projected[mask]-q[mask],axis=1)<=3.1)>.9
        reference=cv2.perspectiveTransform(p.reshape(-1,1,2),gt).reshape(-1,2)
        assert np.median(np.linalg.norm(projected-reference,axis=1))<3.
    H,m=estimate_homography(KP1,KP2,EXPECTED[:3]);assert H is None and m.shape==(3,) and not m.any()
    _must_raise(lambda:estimate_homography(KP1,KP2,EXPECTED,0))
def test_errors():
    before=P.copy(); target=cv2.perspectiveTransform(P.reshape(-1,1,2),GT).reshape(-1,2)
    expected=np.linalg.norm(target-Q,axis=1)
    np.testing.assert_allclose(transfer_errors(P,Q,GT),expected,atol=1e-7)
    np.testing.assert_array_equal(P,before)
    assert transfer_errors(P[:0],Q[:0],GT).shape==(0,)
    _must_raise(lambda:transfer_errors(P,Q[:1],GT))
def test_select():
    expected=REFERENCE_SELECTION
    assert choose_config(VALIDATION_RECORDS)==expected
    for config in sorted({r['config'] for r in VALIDATION_RECORDS if r['split']=='validation'}):
        subset=[r for r in VALIDATION_RECORDS if r['config']==config or r['split']=='test']
        assert choose_config(subset)==config
    assert choose_config(list(reversed(VALIDATION_RECORDS)))==expected
    only=[r for r in VALIDATION_RECORDS if r['split']=='test']
    _must_raise(lambda:choose_config(only))
    _must_raise(lambda:choose_config([]))
raw=0
for title,points,fn in [('ratio',20,test_ratio),('homography',30,test_homography),('errors',25,test_errors),('selection',25,test_select)]:
    try:fn();earned=points;detail='OK'
    except Exception as exc:earned=0;detail=f'{type(exc).__name__}: {exc}'
    raw+=earned;print(f'{title}: {earned}/{points} {detail}')
identity_ok=bool(full_name.strip() and student_group.strip())
if not identity_ok:print('Заполните имя и группу / Fill in name and group')
print(json.dumps({'name':full_name.strip(),'group':student_group.strip(),'assignment':assignment_id,'score':float(raw if identity_ok else 0)},ensure_ascii=False,allow_nan=False))


## Короткий вывод / Short explanation
RU: в 5–8 предложениях сравните число matches и геометрическую точность, объясните назначение RANSAC и ограничение гомографии. Отметьте, какие сцены использовали для настройки. Автотесты проверяют функции, пояснение обсуждается на защите и не даёт автоматических «баллов за стиль».
EN: in 5–8 sentences compare match count and geometric precision, explain RANSAC and a homography limitation. Identify tuning scenes. Tests grade functions; explanations are discussed during review and do not receive automatic style points.